# Day 070 — Solution: Image Generator

In [ ]:
_GENERATOR_SRC = '"""image_generator.py — Day 070: Image Generation.\n\nGenerates images from text prompts using a diffusion model.\n\nSetup (real generation):\n    pip install diffusers transformers accelerate torch\n    # GPU recommended; CPU works but is very slow\n\nUsage:\n    from image_generator import ImageGenerator\n\n    # Quick test (no GPU needed)\n    mock = lambda prompt, **kw: Image.new("RGB", (kw.get("width",512), kw.get("height",512)), "steelblue")\n    gen = ImageGenerator(generate_fn=mock)\n    img = gen.generate("a cat on a moon")\n    img.save("test.png")\n\n    # Real generation (requires diffusers + downloaded model)\n    gen = ImageGenerator(model_id="runwayml/stable-diffusion-v1-5")\n    img = gen.generate("a cat on a moon")\n"""\nimport math\nfrom pathlib import Path\nfrom PIL import Image\n\n# Default style templates — positive/negative prompt pairs per style\nSTYLE_TEMPLATES = {\n    "cinematic": {\n        "positive": "cinematic lighting, film grain, anamorphic lens, dramatic shadows",\n        "negative": "flat lighting, cartoon, illustration, bright colours",\n    },\n    "photorealistic": {\n        "positive": "photorealistic, 8k uhd, high detail, DSLR, sharp focus",\n        "negative": "painting, drawing, blur, watermark, cartoon",\n    },\n    "watercolor": {\n        "positive": "watercolor painting, soft edges, artistic, pastel tones",\n        "negative": "photorealistic, sharp, digital art, 3d render",\n    },\n    "anime": {\n        "positive": "anime style, cel shaded, vibrant colours, clean lines",\n        "negative": "photorealistic, watercolor, oil painting, sketch",\n    },\n}\n\n# Common quality-boost tags to append to positive prompts\nQUALITY_TAGS = ["masterpiece", "best quality", "highly detailed"]\n\n# Common negative-quality tags to append to negative prompts\nNEGATIVE_QUALITY_TAGS = ["blurry", "low quality", "watermark", "text", "cropped"]\n\n\ndef build_prompt(subject: str, style: str = "",\n                 quality_tags: list = None,\n                 negative_tags: list = None) -> dict:\n    """Build a positive/negative prompt pair for image generation.\n\n    Args:\n        subject:       Main subject description\n        style:         Visual style string (e.g. "oil painting, impressionist")\n        quality_tags:  Extra positive quality tags (appended after subject+style)\n        negative_tags: Tags describing what to avoid in the output\n    Returns:\n        dict with \'positive\' (str) and \'negative\' (str) keys\n    """\n    parts = [subject]\n    if style:\n        parts.append(style)\n    if quality_tags:\n        parts.extend(quality_tags)\n    positive = ", ".join(p.strip() for p in parts if p.strip())\n    negative = ", ".join(t.strip() for t in (negative_tags or []) if t.strip())\n    return {"positive": positive, "negative": negative}\n\n\ndef apply_style_template(base_prompt: str, template_name: str,\n                          templates: dict = None) -> dict:\n    """Merge a base prompt with a named style template.\n\n    Args:\n        base_prompt:   Subject or scene description\n        template_name: Key in STYLE_TEMPLATES (or custom templates dict)\n        templates:     Custom template dict; defaults to STYLE_TEMPLATES\n    Returns:\n        dict with \'positive\' (str) and \'negative\' (str) keys\n    Raises:\n        ValueError for unknown template names\n    """\n    tmpl_dict = templates if templates is not None else STYLE_TEMPLATES\n    tmpl = tmpl_dict.get(template_name)\n    if tmpl is None:\n        available = list(tmpl_dict.keys())\n        raise ValueError(\n            f"Unknown style template {template_name!r}. "\n            f"Available: {available}"\n        )\n    pos = base_prompt + ", " + tmpl["positive"] if base_prompt.strip() else tmpl["positive"]\n    return {"positive": pos, "negative": tmpl["negative"]}\n\n\ndef generate_image(prompt: str, negative: str = "",\n                   generate_fn=None,\n                   width: int = 512, height: int = 512,\n                   steps: int = 20, guidance_scale: float = 7.5,\n                   seed: int = 42) -> Image.Image:\n    """Generate an image from a text prompt.\n\n    Args:\n        prompt:         Positive text prompt\n        negative:       Negative text prompt (describe what to avoid)\n        generate_fn:    callable(prompt, **kwargs) -> PIL.Image for testing\n        width:          Output width in pixels (must be multiple of 8)\n        height:         Output height in pixels (must be multiple of 8)\n        steps:          Denoising steps (more steps = higher quality, slower)\n        guidance_scale: CFG scale — how strongly the image follows the prompt\n        seed:           Random seed for reproducibility\n    Returns:\n        PIL.Image.Image\n    """\n    if generate_fn is not None:\n        return generate_fn(prompt, negative=negative, width=width,\n                           height=height, steps=steps,\n                           guidance_scale=guidance_scale, seed=seed)\n    from diffusers import StableDiffusionPipeline\n    import torch\n    pipe = StableDiffusionPipeline.from_pretrained(\n        "runwayml/stable-diffusion-v1-5",\n        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n    )\n    if torch.cuda.is_available():\n        pipe = pipe.to("cuda")\n    generator = torch.Generator().manual_seed(seed)\n    result = pipe(\n        prompt,\n        negative_prompt=negative or None,\n        width=width, height=height,\n        num_inference_steps=steps,\n        guidance_scale=guidance_scale,\n        generator=generator,\n    )\n    return result.images[0]\n\n\ndef generate_variations(base_prompt: str, n_variations: int,\n                         generate_fn=None, **kwargs) -> list:\n    """Generate n_variations of the same prompt, each with a different seed.\n\n    Args:\n        base_prompt:  Text prompt\n        n_variations: Number of images to generate\n        generate_fn:  Callable for testing (see generate_image)\n        **kwargs:     Forwarded to generate_image (width, height, steps, …)\n    Returns:\n        list of PIL.Image.Image, length == n_variations\n    """\n    images = []\n    for i in range(n_variations):\n        seed = i * 1000 + 42\n        img = generate_image(base_prompt, generate_fn=generate_fn, seed=seed, **kwargs)\n        images.append(img)\n    return images\n\n\ndef create_image_grid(images: list, cols: int = 2) -> Image.Image:\n    """Arrange a list of PIL Images in a rectangular grid.\n\n    Args:\n        images: list of PIL.Image.Image — all should have the same size\n        cols:   number of columns in the grid\n    Returns:\n        Single PIL.Image.Image compositing all inputs\n    Raises:\n        ValueError if images is empty\n    """\n    if not images:\n        raise ValueError("images list must not be empty")\n    cols = max(1, min(cols, len(images)))\n    rows = math.ceil(len(images) / cols)\n    w, h = images[0].size\n    grid = Image.new("RGB", (cols * w, rows * h), "white")\n    for i, img in enumerate(images):\n        col = i % cols\n        row = i // cols\n        if img.size != (w, h):\n            img = img.resize((w, h), Image.Resampling.LANCZOS)\n        grid.paste(img, (col * w, row * h))\n    return grid\n\n\nclass ImageGenerator:\n    """Generate images from text prompts using a diffusion model.\n\n    Inject generate_fn for testing without a GPU or installed diffusers::\n\n        mock = lambda p, **kw: Image.new("RGB", (kw.get("width", 512), kw.get("height", 512)), "steelblue")\n        gen = ImageGenerator(generate_fn=mock)\n    """\n\n    def __init__(self, model_id: str = "runwayml/stable-diffusion-v1-5",\n                 generate_fn=None) -> None:\n        self._model_id  = model_id\n        self._generate_fn = generate_fn\n\n    def generate(self, prompt: str, negative: str = "",\n                 width: int = 512, height: int = 512,\n                 steps: int = 20, guidance_scale: float = 7.5,\n                 seed: int = 42) -> Image.Image:\n        """Generate a single image from a prompt."""\n        return generate_image(\n            prompt, negative=negative,\n            generate_fn=self._generate_fn,\n            width=width, height=height,\n            steps=steps, guidance_scale=guidance_scale, seed=seed,\n        )\n\n    def batch(self, prompts: list, **kwargs) -> list:\n        """Generate one image per prompt. Returns list[Image.Image]."""\n        return [self.generate(p, **kwargs) for p in prompts]\n\n    def grid(self, prompts: list, cols: int = 2, **kwargs) -> Image.Image:\n        """Generate images for all prompts and stitch into a grid."""\n        images = self.batch(prompts, **kwargs)\n        return create_image_grid(images, cols=cols)\n'
from pathlib import Path
Path('image_generator.py').write_text(_GENERATOR_SRC, encoding='utf-8')
print('image_generator.py written.')

In [ ]:
import math
from PIL import Image
from image_generator import (
    STYLE_TEMPLATES, build_prompt, apply_style_template,
    generate_image, generate_variations, create_image_grid,
    ImageGenerator,
)

_mock = lambda p, **kw: Image.new('RGB', (kw.get('width', 512), kw.get('height', 512)), 'steelblue')
gen = ImageGenerator(generate_fn=_mock)

# 1. build_prompt
p = build_prompt('a cat', style='oil painting',
                 quality_tags=['masterpiece'], negative_tags=['blurry'])
assert p['positive'] == 'a cat, oil painting, masterpiece'
assert p['negative'] == 'blurry'
print("\u2705 build_prompt correct")

# 2. apply_style_template
r = apply_style_template('a forest', 'cinematic')
assert r['positive'].startswith('a forest, cinematic')
assert r['negative'] == STYLE_TEMPLATES['cinematic']['negative']
print("\u2705 apply_style_template correct")

# 3. generate_image
img = generate_image('a sunset', generate_fn=_mock, width=256, height=128)
assert img.size == (256, 128) and img.mode == 'RGB'
print("\u2705 generate_image returns correct PIL Image")

# 4. generate_variations
imgs = generate_variations('a dog', 3, generate_fn=_mock, width=64, height=64)
assert len(imgs) == 3 and all(isinstance(i, Image.Image) for i in imgs)
print("\u2705 generate_variations returns 3 images")

# 5. create_image_grid: 4 images, 2 cols → (128, 128)
tiles = [Image.new('RGB', (64, 64), 'steelblue') for _ in range(4)]
grid = create_image_grid(tiles, cols=2)
assert grid.size == (128, 128), f"Got {grid.size}"
print("\u2705 create_image_grid 2x2 = (128,128)")

# 6. ImageGenerator.generate
img2 = gen.generate('a cloud', width=128, height=64)
assert img2.size == (128, 64)
print("\u2705 ImageGenerator.generate correct")

# 7. ImageGenerator.batch
prompts = ['a cat', 'a dog']
batch = gen.batch(prompts, width=32, height=32)
assert len(batch) == 2 and batch[0].size == (32, 32)
print("\u2705 ImageGenerator.batch correct")

# 8. ImageGenerator.grid
g = gen.grid(prompts, cols=2, width=32, height=32)
assert g.size == (64, 32), f"Got {g.size}"
print("\u2705 ImageGenerator.grid correct")

print("\nImage Generator complete!")
